In [1]:
import pandas as pd

# Load both files
ingredients = pd.read_csv('recipe_ingredients.csv')
recipes = pd.read_csv('recipes_master.csv')

# Aggregate ingredients with quantity
agg_ingredients = ingredients.groupby('recipe_id').agg(
    ingredients_list=('ingredient_name', lambda x: ', '.join(
        x + ' (' + ingredients.loc[x.index, 'quantity'] + ')'
    )),
    ingredient_count=('ingredient_name', 'count')
).reset_index()

# Merge with recipes
merged = pd.merge(recipes, agg_ingredients, on='recipe_id', how='inner')

merged.to_csv('recipes_ingredients_master.csv', index=False)

In [2]:
import pandas as pd

# Load files
ingredients = pd.read_csv('recipe_ingredients.csv')
carbon = pd.read_excel('Wolfram_Food_Carbon_Footprint.xlsx')

# Clean wolfram ingredients
wolfram_ingredients = set(carbon['Clean Name'].dropna().str.lower().str.strip())

# Clean recipe ingredients
ingredients['ingredient_lower'] = ingredients['ingredient_name'].str.lower().str.strip()

# Manual map
manual_map = {
    'lentils (dal)': 'lentil',
    'mint leaves': 'fresh mint',
    'fresh coriander': 'coriander seed',
    'tamarind paste': 'tamarind',
    'sesame seeds': 'sesame',
    'coriander': 'coriander seed',
    'green chili': 'chili pepper',
    'bell pepper': 'pepper',
    'oil': 'sesame oil',
    'cooking oil': 'sesame oil',
    'salt': 'sea salt',
    'all-purpose flour': 'wheat flour',
    'red chili powder': 'chili pepper',
    'garam masala': 'spice',
    'black pepper': 'black peppercorn',
    'cloves': 'clove',
    'milk': 'milks',
    'water': 'bottled water',
}

ingredients['ingredient_mapped'] = ingredients['ingredient_lower'].replace(manual_map)
ingredients['in_wolfram'] = ingredients['ingredient_mapped'].isin(wolfram_ingredients)

# Match summary per recipe
match_summary = ingredients.groupby('recipe_id').agg(
    total_ingredients=('ingredient_name', 'count'),
    matched_ingredients=('in_wolfram', 'sum')
).reset_index()

match_summary['match_pct'] = (match_summary['matched_ingredients'] / match_summary['total_ingredients'] * 100).round(1)
match_summary['fully_matched'] = match_summary['matched_ingredients'] == match_summary['total_ingredients']

print(f"Total recipes: {len(match_summary)}")
print(f"Fully matched: {match_summary['fully_matched'].sum()}")
print(f"% fully matched: {match_summary['fully_matched'].mean()*100:.1f}%")
print(f"\nMatch % distribution:")
print(match_summary['match_pct'].describe())

Total recipes: 9997
Fully matched: 0
% fully matched: 0.0%

Match % distribution:
count    9997.000000
mean       62.033760
std        12.815696
min        20.000000
25%        53.800000
50%        62.500000
75%        71.400000
max        93.800000
Name: match_pct, dtype: float64


In [3]:
unmatched = ingredients[ingredients['in_wolfram'] == False]
top_unmatched = (unmatched.groupby('ingredient_mapped')
                 .size()
                 .reset_index(name='count')
                 .sort_values('count', ascending=False)
                 .head(20))
print(top_unmatched.to_string())

   ingredient_mapped  count
14             onion  10664
15            paneer   3100
7               eggs   2104
11            lentil   2071
8   fenugreek leaves   2050
6       curry leaves   2004
9               fish   1746
3            chicken   1642
17          semolina   1597
4       chili pepper   1534
19          tamarind   1383
10              ghee   1353
12             milks   1330
13     mustard seeds   1138
18            sesame   1115
0           bay leaf    814
5              clove    811
1             carrot    761
16            potato    755
2        cauliflower    741


In [4]:
import pandas as pd

# Load files
ingredients = pd.read_csv('recipe_ingredients.csv')
carbon = pd.read_excel('Wolfram_Food_Carbon_Footprint.xlsx')

# Clean wolfram ingredients
wolfram_ingredients = set(carbon['Clean Name'].dropna().str.lower().str.strip())

# Clean recipe ingredients
ingredients['ingredient_lower'] = ingredients['ingredient_name'].str.lower().str.strip()

manual_map = {
    # previous mappings
    'lentils (dal)': 'lentil',
    'mint leaves': 'fresh mint',
    'fresh coriander': 'coriander seed',
    'tamarind paste': 'tamarind',
    'sesame seeds': 'sesame',
    'coriander': 'coriander seed',
    'green chili': 'chili pepper',
    'bell pepper': 'pepper',
    'oil': 'sesame oil',
    'cooking oil': 'sesame oil',
    'salt': 'sea salt',
    'all-purpose flour': 'wheat flour',
    'red chili powder': 'chili pepper',
    'garam masala': 'spice',
    'black pepper': 'black peppercorn',
    'cloves': 'clove',
    'milk': 'milks',
    'water': 'bottled water',
    # new mappings
    'eggs': 'egg',
    'fenugreek leaves': 'fenugreek seed',
    'curry leaves': 'curry powder',
    'chickpeas': 'chickpea',
    'fish': 'fishes',
    'ghee': 'butter',
    'mustard seeds': 'mustard',
    'almonds': 'almond',
    'cashews': 'cashew',
    'raisins': 'raisin',
    'bay leaf': 'herb',
    'peas': 'pea',
}

ingredients['ingredient_mapped'] = ingredients['ingredient_lower'].replace(manual_map)
ingredients['in_wolfram'] = ingredients['ingredient_mapped'].isin(wolfram_ingredients)

# Match summary per recipe
match_summary = ingredients.groupby('recipe_id').agg(
    total_ingredients=('ingredient_name', 'count'),
    matched_ingredients=('in_wolfram', 'sum')
).reset_index()

match_summary['match_pct'] = (match_summary['matched_ingredients'] / match_summary['total_ingredients'] * 100).round(1)
match_summary['fully_matched'] = match_summary['matched_ingredients'] == match_summary['total_ingredients']

print(f"Total recipes: {len(match_summary)}")
print(f"Fully matched: {match_summary['fully_matched'].sum()}")
print(f"% fully matched: {match_summary['fully_matched'].mean()*100:.1f}%")
print(f"\nMatch % distribution:")
print(match_summary['match_pct'].describe())

Total recipes: 9997
Fully matched: 0
% fully matched: 0.0%

Match % distribution:
count    9997.000000
mean       66.142703
std        12.281356
min        21.400000
25%        57.100000
50%        66.700000
75%        75.000000
max        94.400000
Name: match_pct, dtype: float64


In [5]:
unmatched = ingredients[ingredients['in_wolfram'] == False]
top_unmatched = (unmatched.groupby('ingredient_mapped')
                 .size()
                 .reset_index(name='count')
                 .sort_values('count', ascending=False)
                 .head(20))
print(top_unmatched.to_string())

   ingredient_mapped  count
11             onion  10664
12            paneer   3100
9             lentil   2071
5           chickpea   1994
4            chicken   1642
16          semolina   1597
6       chili pepper   1534
18          tamarind   1383
10             milks   1330
0             almond   1116
17            sesame   1115
2             cashew   1109
15            raisin   1069
8               herb    814
7              clove    811
1             carrot    761
13               pea    758
14            potato    755
3        cauliflower    741
19            tomato    672


In [6]:
import pandas as pd
import requests
from io import StringIO

# =========================================================
# STEP 1: LOAD OWID DATASET
# =========================================================
def load_owid():
    url = "https://ourworldindata.org/grapher/ghg-per-kg-poore.csv?v=1&csvType=full"
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    response = session.get(url, timeout=20)
    response.raise_for_status()
    df = pd.read_csv(StringIO(response.text))
    latest_year = df["Year"].max()
    df = df[df["Year"] == latest_year].copy()
    co2_col = [c for c in df.columns if c not in {"Entity", "Code", "Year"}][0]
    return df, co2_col

owid_df, co2_col = load_owid()

# Ingredients to fetch from OWID (not in Wolfram)
owid_map = {
    'paneer':   'Cheese',
    'semolina': 'Wheat & Rye',
}

# Build OWID CO2 lookup
owid_co2 = {}
for ingredient, entity in owid_map.items():
    row = owid_df[owid_df["Entity"].str.lower() == entity.lower()]
    if not row.empty:
        owid_co2[ingredient] = round(float(row.iloc[0][co2_col]), 2)

print("OWID values fetched:")
for k, v in owid_co2.items():
    print(f"  {k}: {v} kg CO2e/kg")

# =========================================================
# STEP 2: LOAD FILES
# =========================================================
ingredients = pd.read_csv('recipe_ingredients.csv')
carbon = pd.read_excel('Wolfram_Food_Carbon_Footprint.xlsx')

# =========================================================
# STEP 3: WOLFRAM MATCHING (only ingredients in Wolfram)
# =========================================================
wolfram_ingredients = set(carbon['Clean Name'].dropna().str.lower().str.strip())
ingredients['ingredient_lower'] = ingredients['ingredient_name'].str.lower().str.strip()

manual_map = {
    'lentils (dal)':    'lentil',
    'mint leaves':      'fresh mint',
    'fresh coriander':  'coriander seed',
    'tamarind paste':   'tamarind',
    'sesame seeds':     'sesame',
    'coriander':        'coriander seed',
    'green chili':      'chili pepper',
    'bell pepper':      'pepper',
    'oil':              'sesame oil',
    'cooking oil':      'sesame oil',
    'all-purpose flour':'wheat flour',
    'red chili powder': 'chili pepper',
    'garam masala':     'spice',
    'black pepper':     'black peppercorn',
    'cloves':           'clove',
    'milk':             'milks',
    'water':            'bottled water',
    'eggs':             'egg',
    'fenugreek leaves': 'fenugreek seed',
    'curry leaves':     'curry powder',
    'chickpeas':        'chickpea',
    'fish':             'fishes',
    'mustard seeds':    'mustard',
    'almonds':          'almond',
    'cashews':          'cashew',
    'raisins':          'raisin',
    'peas':             'pea',
    'ghee':             'butter',
    'salt':             'sea salt',
}

ingredients['ingredient_mapped'] = ingredients['ingredient_lower'].replace(manual_map)
ingredients['in_wolfram'] = ingredients['ingredient_mapped'].isin(wolfram_ingredients)

# =========================================================
# STEP 4: OWID FALLBACK
# =========================================================
ingredients['in_owid'] = (
    ~ingredients['in_wolfram'] &
    ingredients['ingredient_lower'].isin(owid_co2.keys())
)

ingredients['matched'] = ingredients['in_wolfram'] | ingredients['in_owid']

# =========================================================
# STEP 5: MATCH SUMMARY
# =========================================================
match_summary = ingredients.groupby('recipe_id').agg(
    total_ingredients=('ingredient_name', 'count'),
    matched_ingredients=('matched', 'sum')
).reset_index()

match_summary['match_pct'] = (match_summary['matched_ingredients'] / match_summary['total_ingredients'] * 100).round(1)
match_summary['fully_matched'] = match_summary['matched_ingredients'] == match_summary['total_ingredients']

print(f"\nTotal recipes: {len(match_summary)}")
print(f"Fully matched: {match_summary['fully_matched'].sum()}")
print(f"% fully matched: {match_summary['fully_matched'].mean()*100:.1f}%")
print(f"\nMatch % distribution:")
print(match_summary['match_pct'].describe())


OWID values fetched:
  paneer: 23.88 kg CO2e/kg
  semolina: 1.57 kg CO2e/kg

Total recipes: 9997
Fully matched: 0
% fully matched: 0.0%

Match % distribution:
count    9997.000000
mean       70.491988
std        11.717119
min        25.000000
25%        62.500000
50%        71.400000
75%        80.000000
max        94.700000
Name: match_pct, dtype: float64


In [7]:
print(match_summary.head(10).to_string())

  recipe_id  total_ingredients  matched_ingredients  match_pct  fully_matched
0  RCP00001                 11                    9       81.8          False
1  RCP00002                  7                    6       85.7          False
2  RCP00003                  8                    5       62.5          False
3  RCP00004                  8                    6       75.0          False
4  RCP00005                 12                    9       75.0          False
5  RCP00006                 12                    9       75.0          False
6  RCP00007                  9                    6       66.7          False
7  RCP00008                  7                    5       71.4          False
8  RCP00009                  7                    5       71.4          False
9  RCP00010                  7                    6       85.7          False


In [8]:
recipes = pd.read_csv('recipes_master.csv')

result = ingredients.merge(recipes[['recipe_id', 'recipe_name']], on='recipe_id', how='left')

print(result[['recipe_id', 'recipe_name', 'ingredient_name', 'matched']].head(10).to_string())

  recipe_id recipe_name ingredient_name  matched
0  RCP00001       Sajji           Onion    False
1  RCP00001       Sajji          Garlic     True
2  RCP00001       Sajji            Salt     True
3  RCP00001       Sajji          Shrimp     True
4  RCP00001       Sajji         Chicken    False
5  RCP00001       Sajji             Oil     True
6  RCP00001       Sajji     Wheat Flour     True
7  RCP00001       Sajji           Sugar     True
8  RCP00001       Sajji   Mustard Seeds     True
9  RCP00001       Sajji          Mutton     True


In [9]:
import pandas as pd
import requests
from io import StringIO

# Build the complete ingredient CO₂e lookup.
# Wolfram (primary) is queried first; OWID (fallback, via API) supplies the few
# foods Wolfram does not cover. 100% of the 58 canonical ingredients are covered.

# ── Wolfram (primary) ──────────────────────────────────────────────────────
carbon = pd.read_excel('Wolfram_Food_Carbon_Footprint.xlsx')
wolfram_co2 = dict(zip(
    carbon['Clean Name'].dropna().str.lower().str.strip(),
    carbon['CO2e Avg (kg)']
))

# ── OWID (fallback) — fetched via API, latest year ─────────────────────────
def load_owid():
    url = "https://ourworldindata.org/grapher/ghg-per-kg-poore.csv?v=1&csvType=full"
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    resp = session.get(url, timeout=20)
    resp.raise_for_status()
    df = pd.read_csv(StringIO(resp.text))
    df = df[df["Year"] == df["Year"].max()]
    col = [c for c in df.columns if c not in {"Entity", "Code", "Year"}][0]
    return {row["Entity"].lower(): float(row[col]) for _, row in df.iterrows()}

owid_co2_raw = load_owid()

# Wolfram alias map. Targets use the exact inflection present in the Wolfram
# file (the file pluralises most foods). An earlier version of this map pointed
# several names at a singular form absent from the file (e.g. 'almond',
# 'cashew', 'milks', 'chili pepper'), which silently routed those ingredients to
# OWID despite Wolfram having them. Those targets are corrected here so Wolfram
# supplies the value as intended; only 4 ingredients now fall back to OWID.
WOLFRAM_MAP = {
    'lentils (dal)':    'lentils',       'mint leaves':      'fresh mint',
    'fresh coriander':  'coriander seed','tamarind paste':   'tamarinds',
    'sesame seeds':     'sesames',       'coriander':        'coriander seed',
    'green chili':      'chili peppers', 'bell pepper':      'pepper',
    'oil':              'sesame oil',    'cooking oil':      'sesame oil',
    'all-purpose flour':'wheat flour',   'red chili powder': 'chili peppers',
    'garam masala':     'spice',         'black pepper':     'black peppercorn',
    'cloves':           'cloves',        'milk':             'milk',
    'water':            'bottled water', 'eggs':             'egg',
    'fenugreek leaves': 'fenugreek seed','curry leaves':     'curry powder',
    'chickpeas':        'chickpeas',     'fish':             'fishes',
    'mustard seeds':    'mustard',       'almonds':          'almonds',
    'cashews':          'cashews',       'raisins':          'raisins',
    'bay leaf':         'herbs',         'peas':             'peas',
    'ghee':             'butter',        'salt':             'sea salt',
    'carrot':           'carrots',       'cauliflower':      'cauliflowers',
    'chicken':          'chickens',      'onion':            'onions',
}

# OWID fallback — only these 4 ingredients are absent from the Wolfram file.
OWID_MAP = {
    'paneer':   'cheese',
    'semolina': 'wheat & rye',   # proxy: refined semolina ~ whole-grain wheat;
                                 # OWID has no refined-semolina category.
    'potato':   'potatoes',
    'tomato':   'tomatoes',
}

ingr_df = pd.read_csv('recipe_ingredients.csv')
ingr_df['ingredient_lower'] = ingr_df['ingredient_name'].str.lower().str.strip()

rows, n_wolfram, n_owid = [], 0, 0
for ingr in ingr_df['ingredient_lower'].unique():
    wolfram_key = WOLFRAM_MAP.get(ingr, ingr)
    co2 = wolfram_co2.get(wolfram_key)            # try Wolfram first
    source = 'wolfram'
    if co2 is None:
        owid_entity = OWID_MAP.get(ingr)
        if owid_entity:
            co2 = round(owid_co2_raw.get(owid_entity), 2)  # OWID fallback
            source = 'owid'
    if co2 is not None:
        rows.append({'ingredient': ingr, 'co2_kg_per_kg': float(co2)})
        n_wolfram += source == 'wolfram'
        n_owid += source == 'owid'

ingredient_co2 = pd.DataFrame(rows)
ingredient_co2.to_csv('ingredient_co2.csv', index=False)

n = ingr_df['ingredient_lower'].nunique()
print(f"Saved ingredient_co2.csv: {len(ingredient_co2)}/{n} ingredients "
      f"({len(ingredient_co2)/n*100:.0f}% coverage)")
print(f"  Wolfram-sourced : {n_wolfram}")
print(f"  OWID-sourced    : {n_owid}  ({sorted(OWID_MAP)})")
print()
for _, r in ingredient_co2.sort_values('ingredient').iterrows():
    print(f"  {r['ingredient']:<20} {r['co2_kg_per_kg']:.3f} kg CO₂e/kg")


Saved ingredient_co2.csv: 58/58 ingredients (100% coverage)
  Wolfram-sourced : 54
  OWID-sourced    : 4  (['paneer', 'potato', 'semolina', 'tomato'])

  all-purpose flour    0.700 kg CO₂e/kg
  almonds              2.150 kg CO₂e/kg
  basmati rice         4.100 kg CO₂e/kg
  bay leaf             1.200 kg CO₂e/kg
  beef                 59.850 kg CO₂e/kg
  bell pepper          2.000 kg CO₂e/kg
  black pepper         8.100 kg CO₂e/kg
  butter               14.350 kg CO₂e/kg
  cardamom             8.100 kg CO₂e/kg
  carrot               0.550 kg CO₂e/kg
  cashews              2.350 kg CO₂e/kg
  cauliflower          0.800 kg CO₂e/kg
  chicken              5.550 kg CO₂e/kg
  chickpea flour       0.900 kg CO₂e/kg
  chickpeas            1.150 kg CO₂e/kg
  cinnamon             8.100 kg CO₂e/kg
  cloves               8.100 kg CO₂e/kg
  coconut milk         0.600 kg CO₂e/kg
  coriander            1.050 kg CO₂e/kg
  cream                5.000 kg CO₂e/kg
  cumin                8.100 kg CO₂e/kg
  curr

In [10]:
# Load the dataset and check the cuisine origin distribution
import pandas as pd
recipes = pd.read_csv('recipes_master.csv')
cuisine_counts = recipes['cuisine'].value_counts()
print("Cuisine origin distribution:")
print(cuisine_counts)
# Get percentage distribution
cuisine_pct = recipes['cuisine'].value_counts(normalize=True) * 100
print("\nCuisine origin percentage distribution:")  
print(cuisine_pct.round(2))


Cuisine origin distribution:
cuisine
Indian         3498
Pakistani      3000
Bangladeshi    1500
Afghan         1000
Fusion          999
Name: count, dtype: int64

Cuisine origin percentage distribution:
cuisine
Indian         34.99
Pakistani      30.01
Bangladeshi    15.00
Afghan         10.00
Fusion          9.99
Name: proportion, dtype: float64
